# ARC Prize 2026 — ARC-AGI-3 Agent

A Kaggle code-competition starter notebook. The baseline uses the official ARC-AGI-3 agent framework and competition gateway. It resets games when needed, stops only after a win, and explores available actions in a reproducible order. This is an integration baseline, not a high-scoring solver; improve `MyAgent.choose_action` using observations and evidence gathered within each game.

Internet is disabled. The ARC package is installed from the competition-provided wheel directory. The gateway creates the real `submission.parquet` during the competition rerun; a small placeholder is emitted during the notebook's validation run.

In [ ]:
# Install only from the competition's attached offline wheelhouse.
from pathlib import Path
wheelhouse = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
if not wheelhouse.is_dir():
    raise FileNotFoundError(f'Required competition wheelhouse not found: {wheelhouse}')
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
from __future__ import annotations

import hashlib
from typing import Any

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class MyAgent(Agent):
    """A reproducible bounded-exploration baseline for ARC-AGI-3."""

    MAX_ACTIONS = 80

    @property
    def name(self) -> str:
        return f'{super().name}.{self.MAX_ACTIONS}'

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    @staticmethod
    def _legal_action_names(latest_frame: FrameData) -> set[str]:
        available = getattr(latest_frame, 'available_actions', None)
        if not available:
            return set()
        names: set[str] = set()
        for item in available:
            names.add(str(getattr(item, 'name', item)).upper())
            names.add(str(getattr(item, 'value', item)).upper())
        return names

    @staticmethod
    def _grid_size(latest_frame: FrameData) -> tuple[int, int]:
        """Infer coordinate bounds from the observed frame; use 64x64 if absent."""
        frame: Any = getattr(latest_frame, 'frame', None)
        try:
            plane = frame[0] if frame and isinstance(frame[0], (list, tuple)) else frame
            height = len(plane)
            width = len(plane[0]) if height else 0
            if height > 0 and width > 0:
                return width, height
        except (TypeError, IndexError):
            pass
        return 64, 64

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET

        legal_names = self._legal_action_names(latest_frame)
        candidates = [
            action for action in GameAction
            if action is not GameAction.RESET
            and (not legal_names or action.name.upper() in legal_names
                 or str(action.value).upper() in legal_names)
        ]
        if not candidates:
            raise RuntimeError('The current observation exposes no usable game actions.')

        # Rotate deterministically through candidate actions, changing the
        # starting point when the visible state changes. Repeated states still
        # make progress through the action list instead of repeating one move.
        visible = repr(getattr(latest_frame, 'frame', None))
        state_hash = int(hashlib.sha256(visible.encode('utf-8')).hexdigest()[:8], 16)
        index = (len(frames) + state_hash) % len(candidates)
        action = candidates[index]
        if action.is_complex():
            width, height = self._grid_size(latest_frame)
            action.set_data({'x': width // 2, 'y': height // 2})
            action.reasoning = {'why': 'bounded baseline exploration at observed grid center'}
        else:
            action.reasoning = f'bounded baseline exploration: {action.value}'
        return action

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the competition gateway to accept requests.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games

    framework = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents')
    if not framework.is_dir():
        raise FileNotFoundError(f'Required ARC-AGI-3 agent framework not found: {framework}')
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /tmp/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register only the lightweight agents needed by this submission.
    agents_init = '''from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\n\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {\"random\": Random, \"myagent\": MyAgent}\n'''
    Path('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py').write_text(agents_init)
    env_text = '''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n'''
    Path('/kaggle/working/ARC-AGI-3-Agents/.env').write_text(env_text)
    %cd /kaggle/working/ARC-AGI-3-Agents
    !MPLBACKEND=agg python main.py --agent myagent
else:
    # Kaggle validates the committed notebook before the competition rerun.
    # The gateway generates the real submission during the competition rerun.
    import pandas as pd
    validation_output = pd.DataFrame(
        [['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    )
    validation_output.to_parquet('/kaggle/working/submission.parquet', index=False)
    display(validation_output)